In [1]:
from pyspark.sql import SparkSession, types

In [2]:
spark = (
    SparkSession.builder
        .appName("CryptoETL")
        .config("spark.master", "spark://spark-master:7077")
        # ---- Iceberg + Hive Catalog ----
        .config("spark.sql.catalog.hive_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.hive_catalog.catalog-impl", "org.apache.iceberg.hive.HiveCatalog")
        .config("spark.sql.catalog.hive_catalog.uri", "thrift://hive-metastore:9083")
        .config("spark.sql.catalog.hive_catalog.warehouse", "s3a://crypto-data-lake/")
        # ---- Default catalog
        .config("spark.sql.defaultCatalog", "hive_catalog")
        # ---- S3 (MinIO) ----
        .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
        .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        # ---- Iceberg Extensions ----
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
        # ---- Extra JARs ----
        .config("spark.jars", ",".join([
            "/opt/spark-extra-jars/iceberg-spark-runtime-3.5_2.12-1.6.1.jar",
            "/opt/spark-extra-jars/hadoop-aws-3.3.4.jar",
            "/opt/spark-extra-jars/aws-java-sdk-bundle-1.12.262.jar"
        ]))
        .getOrCreate()
)

25/10/05 05:49:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [75]:
landing_date = '2025-10-03'
symbol = 'BTCUSDT'
sql_stmt = f"""
select * from serving_db.klines
where landing_date = DATE('{landing_date}') AND symbol = '{symbol}'
"""

In [76]:
df_sorted = (
    spark.sql(sql_stmt)
    .coalesce(1)  # one partition, not shuffle
    .sortWithinPartitions("group_id")
)

In [77]:
schema = types.StructType(
    [
        *df_sorted.schema.fields,  # keep all original fields
        types.StructField("ema7", types.DoubleType(), True),
        types.StructField("ema20", types.DoubleType(), True),
    ]
)

In [79]:
from pyspark.errors.exceptions.base import AnalysisException
def table_exists(spark: SparkSession, database: str, table: str) -> bool:
    try:
        spark.catalog.getTable(f"{database}.{table}")
        return True
    except AnalysisException:
        return False

def round_half_up(x, decimals=2):
    if x is None:
        return None
    factor = 10**decimals
    return float(int(x * factor + 0.5)) / factor

def calc_ema(value, state):
    if value is None:
        return None
    prev, buffer, period, k = (
        state["prev"],
        state["buffer"],
        state["period"],
        state["k"],
    )
    if prev is None:
        buffer.append(value)
        if len(buffer) == period:
            ema = sum(buffer) / len(buffer)
        else:
            ema = None
    else:
        ema = (value - prev) * k + prev

    state["prev"] = ema
    return ema

def make_ema_in_chunks(prev_ema7, prev_ema20):
    def ema_in_chunks(iterator):
        ema_configs = {
            "ema7": {"period": 7, "k": 2 / (7 + 1), "prev": prev_ema7, "buffer": []},
            "ema20": {"period": 20, "k": 2 / (20 + 1), "prev": prev_ema20, "buffer": []},
        }
    
        for pdf in iterator:
            ema7, ema20 = [], []
            for p in pdf["close_price"]:
                price = float(p)
                e7 = calc_ema(price, ema_configs["ema7"])
                ema7.append(round_half_up(e7, 2) if e7 is not None else None)
                e20 = calc_ema(price, ema_configs["ema20"])
                ema20.append(round_half_up(e20, 2) if e20 is not None else None)
    
            pdf["ema7"] = ema7
            pdf["ema20"] = ema20
            pdf = pdf[[*pdf.columns[:-2], "ema7", "ema20"]]
            yield pdf
    print(f"prev_ema7: {prev_ema7}, prev_ema20: {prev_ema20}")
    return ema_in_chunks

In [80]:
if table_exists(spark, "serving_db", "pattern_two"):
    sql_stmt = f"""
    select ema7, ema20 from serving_db.pattern_two
    where landing_date = date_sub(DATE('{landing_date}'), 1) AND symbol = '{symbol}'
    order by group_id desc
    limit 1
    """
    row = spark.sql(sql_stmt).first()
    prev_ema7, prev_ema20 = (row["ema7"], row["ema20"]) if row else (None, None)
else:
    prev_ema7, prev_ema20 = None, None

ema_in_chunks_with_state = make_ema_in_chunks(prev_ema7, prev_ema20)

prev_ema7: 120429.06, prev_ema20: 120358.33


In [81]:
df = df_sorted.mapInPandas(ema_in_chunks_with_state, schema)

In [82]:
df.createOrReplaceTempView("temp")

In [83]:
df = spark.sql("""
with cte as (
    select
        *,
        case 
            when ema7 > ema20 then 'uptrend' 
            when ema7 < ema20 then 'downtrend' 
            else NULL 
        end as trend,
        LAG(open_price, 1) over(order by group_id) as open_price_prev,
        LAG(close_price, 1) over(order by group_id) as close_price_prev
    from temp
)
select
    group_id,
    group_date,
    open_time,
    open_price,
    high_price,
    low_price,
    close_price,
    volume,
    close_time,
    landing_date,
    symbol,
    ema7,
    ema20,
    trend,
    case 
        when close_price_prev < open_price_prev
            and close_price > open_price
            and open_price < close_price_prev
            and close_price > open_price_prev
            and trend = 'downtrend'
        then 'bullish engulfing'
        when close_price_prev > open_price_prev
            and close_price < open_price
            and open_price > close_price_prev
            and close_price < open_price_prev
            and trend = 'uptrend'
        then 'bearish engulfing'
        else NULL
    end as pattern
from cte
""")

In [84]:
df.show()

25/10/05 05:24:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/05 05:24:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/05 05:24:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+--------+-------------------+----------------+----------+----------+---------+-----------+------+----------------+------------+-------+---------+---------+---------+-----------------+
|group_id|         group_date|       open_time|open_price|high_price|low_price|close_price|volume|      close_time|landing_date| symbol|     ema7|    ema20|    trend|          pattern|
+--------+-------------------+----------------+----------+----------+---------+-----------+------+----------------+------------+-------+---------+---------+---------+-----------------+
| 1954944|2025-10-03 00:00:00|1759449600000253| 120529.35|  120550.9|120106.49|  120156.53|475.09|1759450499707850|  2025-10-03|BTCUSDT|120360.93|120339.11|  uptrend|             NULL|
| 1954945|2025-10-03 00:15:00|1759450500343199| 120156.53| 120297.91| 120079.0|  120279.46|  92.3|1759451399900892|  2025-10-03|BTCUSDT|120340.56|120333.43|  uptrend|             NULL|
| 1954946|2025-10-03 00:30:00|1759451400392272| 120279.46| 120279.47|120163

In [86]:
if table_exists(spark, "serving_db", "pattern_two"):
    df.writeTo("serving_db.pattern_two").overwritePartitions()
else:
    df.writeTo("serving_db.pattern_two").tableProperty(
    "format-version", "2"
    ).partitionedBy("symbol", "landing_date").createOrReplace()

25/10/05 05:24:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/05 05:24:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [85]:
df.select("group_id", "group_date", "landing_date", "symbol", "ema7").show()

+--------+-------------------+------------+-------+---------+
|group_id|         group_date|landing_date| symbol|     ema7|
+--------+-------------------+------------+-------+---------+
| 1954944|2025-10-03 00:00:00|  2025-10-03|BTCUSDT|120360.93|
| 1954945|2025-10-03 00:15:00|  2025-10-03|BTCUSDT|120340.56|
| 1954946|2025-10-03 00:30:00|  2025-10-03|BTCUSDT|120306.97|
| 1954947|2025-10-03 00:45:00|  2025-10-03|BTCUSDT|120263.42|
| 1954948|2025-10-03 01:00:00|  2025-10-03|BTCUSDT|120253.88|
| 1954949|2025-10-03 01:15:00|  2025-10-03|BTCUSDT|120220.22|
| 1954950|2025-10-03 01:30:00|  2025-10-03|BTCUSDT|120190.19|
| 1954951|2025-10-03 01:45:00|  2025-10-03|BTCUSDT|120111.57|
| 1954952|2025-10-03 02:00:00|  2025-10-03|BTCUSDT|120116.81|
| 1954953|2025-10-03 02:15:00|  2025-10-03|BTCUSDT|120084.06|
| 1954954|2025-10-03 02:30:00|  2025-10-03|BTCUSDT|120082.17|
| 1954955|2025-10-03 02:45:00|  2025-10-03|BTCUSDT|120101.08|
| 1954956|2025-10-03 03:00:00|  2025-10-03|BTCUSDT|120148.17|
| 195495

In [88]:
!jupyter nbconvert --to script end_transform_job_pattern_two.ipynb

[NbConvertApp] Converting notebook end_transform_job_pattern_two.ipynb to script
[NbConvertApp] Writing 6225 bytes to end_transform_job_pattern_two.py


In [7]:
landing_date = '2025-10-04'
symbol = 'BTCUSDT'
sql_stmt = f"""
    select group_id, group_date, landing_date, symbol, ema7, ema20 from serving_db.pattern_two
    where landing_date = DATE('{landing_date}') AND symbol = '{symbol}'
    """
spark.sql(sql_stmt).show()

+--------+-------------------+------------+-------+---------+---------+
|group_id|         group_date|landing_date| symbol|     ema7|    ema20|
+--------+-------------------+------------+-------+---------+---------+
| 1955040|2025-10-04 00:00:00|  2025-10-04|BTCUSDT|122290.92|122320.87|
| 1955041|2025-10-04 00:15:00|  2025-10-04|BTCUSDT| 122260.3|122306.36|
| 1955042|2025-10-04 00:30:00|  2025-10-04|BTCUSDT|122209.01|122282.43|
| 1955043|2025-10-04 00:45:00|  2025-10-04|BTCUSDT|122181.76|122265.06|
| 1955044|2025-10-04 01:00:00|  2025-10-04|BTCUSDT|122158.82|122248.39|
| 1955045|2025-10-04 01:15:00|  2025-10-04|BTCUSDT|122106.62|122219.97|
| 1955046|2025-10-04 01:30:00|  2025-10-04|BTCUSDT|122050.95|122187.97|
| 1955047|2025-10-04 01:45:00|  2025-10-04|BTCUSDT|122004.51|122157.23|
| 1955048|2025-10-04 02:00:00|  2025-10-04|BTCUSDT|121971.31|122130.03|
| 1955049|2025-10-04 02:15:00|  2025-10-04|BTCUSDT|121929.74|122099.08|
| 1955050|2025-10-04 02:30:00|  2025-10-04|BTCUSDT|121885.46|122